In [ ]:
import pandas as pd
import soccerdata as sd

print("Starting scrape...", flush=True)

leagues = [
    "ENG-Premier League",
    "ESP-La Liga", 
    "ITA-Serie A",
    "GER-Bundesliga",
    "FRA-Ligue 1",
]

for league in leagues:
    print(f"Trying {league}...", flush=True)
    try:
        u = sd.Understat(leagues=[league], seasons=["2025"], no_cache=True)
        df = u.read_schedule().reset_index()
        print(f"  Got {len(df)} rows", flush=True)
    except Exception as e:
        print(f"  Error: {e}", flush=True)

print("Done.", flush=True)

In [ ]:
import sqlite3
import pandas as pd
import soccerdata as sd
from pathlib import Path

DB_PATH    = Path(r"C:\Users\rhkha\Documents\Documents\Schoolwork\Projects\FIFA-WORLDCUP-PREDICTION\v4_historical_data.sqlite")
NEW_SEASON = "2526"
SCRAPE_SEASON = "2025"

TARGET_LEAGUES = [
    "ENG-Premier League",
    "ESP-La Liga",
    "ITA-Serie A",
    "GER-Bundesliga",
    "FRA-Ligue 1",
]

COLUMN_MAP = {
    "home_team"  : ["home_team", "home"],
    "away_team"  : ["away_team", "away"],
    "home_goals" : ["home_goals", "home_goal", "score_home"],
    "away_goals" : ["away_goals", "away_goal", "score_away"],
    "home_xg"    : ["home_xg", "xg_home", "xgh"],
    "away_xg"    : ["away_xg", "xg_away", "xga"],
    "is_finished": ["is_result", "finished", "status"],
}

def resolve_column(df, candidates):
    for name in candidates:
        if name in df.columns:
            return name
    return None

# Check existing data
conn = sqlite3.connect(DB_PATH)
before = pd.read_sql("SELECT season, COUNT(*) as n FROM matches_xg GROUP BY season ORDER BY season", conn)
conn.close()
print("Existing database:")
print(before.to_string(index=False))
print()

# Scrape
all_dfs = []
for league in TARGET_LEAGUES:
    print(f"Scraping {league}...", flush=True)
    u = sd.Understat(leagues=[league], seasons=[SCRAPE_SEASON], no_cache=True)
    df_raw = u.read_schedule().reset_index()
    print(f"  {len(df_raw)} rows", flush=True)
    all_dfs.append(df_raw)

df = pd.concat(all_dfs, ignore_index=True)

# Resolve columns
rename = {}
for standard, aliases in COLUMN_MAP.items():
    found = resolve_column(df, aliases)
    if found:
        rename[found] = standard

df = df.rename(columns=rename)

# Filter to finished matches
if "is_finished" in df.columns:
    df = df[df["is_finished"] == True].copy()
else:
    df = df.dropna(subset=["home_xg", "away_xg"]).copy()

# Standardise
for std, aliases in [("league",["league"]),("season",["season"]),("date",["date","datetime"])]:
    if std not in df.columns:
        found = resolve_column(df, aliases)
        if found:
            df = df.rename(columns={found: std})

required = ["league","season","date","home_team","away_team","home_goals","away_goals","home_xg","away_xg"]
df_clean = df[required].copy()
df_clean["date"]       = pd.to_datetime(df_clean["date"], errors="coerce")
df_clean["home_goals"] = pd.to_numeric(df_clean["home_goals"], errors="coerce")
df_clean["away_goals"] = pd.to_numeric(df_clean["away_goals"], errors="coerce")
df_clean["home_xg"]    = pd.to_numeric(df_clean["home_xg"],    errors="coerce")
df_clean["away_xg"]    = pd.to_numeric(df_clean["away_xg"],    errors="coerce")
df_clean = df_clean.dropna()

# Override season label to our short code
df_clean["season"] = NEW_SEASON

print(f"\nFinished matches with real xG: {len(df_clean):,}")
print(df_clean.groupby(["league","season"]).size().reset_index(name="matches").to_string(index=False))

# Append to database
conn = sqlite3.connect(DB_PATH)
df_clean.to_sql("matches_xg", conn, if_exists="append", index=False)
after = pd.read_sql("SELECT season, COUNT(*) as n FROM matches_xg GROUP BY season ORDER BY season", conn)
conn.close()

print("\nDatabase after append:")
print(after.to_string(index=False))
print(f"\n✅ {NEW_SEASON} appended successfully.")